# 08 · ASR — Whisper large-v3

Hộp trong sơ đồ offline: **`Videos → Audio → ASR (Whisper large-v3) → asr_result`**

`ffmpeg -vn -ac 1 -ar 16000` tách tiếng khỏi `.mp4` → WAV 16 kHz mono → Whisper.

**Không cắt keyframe.** ASR đọc thẳng file video, đây là nhánh duy nhất không đi qua
`Keyframes` trên sơ đồ.

## Vì sao tuyến này đáng giá

Nó thắng đúng chỗ OCR thua. Câu R1-17 cần dòng chữ trên phông cao 6–8 px, OCR đọc được
nhiều nhất 44% — nhưng người dẫn **đọc nguyên câu đó lên**.

Cầu nối về hệ thống là một phép nhân: `frame = giây × fps`, fps lấy từ bảng BTC.
Giới hạn: ASR chỉ thu hẹp xuống **video + khoảng thời gian**, không chỉ ra frame — là
tuyến lọc, không phải tuyến trả lời.

Chi phí ~3–5 phút cho video 21 phút trên T4 — rẻ nhất trong các tuyến offline.

**Người phụ trách: ai rảnh GPU**


> **Chạy trong một phiên Colab RIÊNG.** Thư viện giữa các notebook xung đột nhau
> (paddleocr/vietocr hạ cấp Pillow), nên đừng chạy hai notebook trong cùng một runtime.

**Sửa đúng một ô: `SECTION 02a`.** Bốn thứ: `PERSON` · `VERSION` · `VIDEO_LIST` ·
`MAX_VIDEOS`. Mọi đường dẫn tự suy ra từ đó.

Mỗi người làm việc độc lập trên thư mục riêng của mình, nên **không cần phối hợp chọn
video với nhau** — trùng video cũng không sao, đầu ra nằm ở hai chỗ khác nhau.

```
/content/drive/MyDrive/aic26/
├── fps_map.json                     ← dùng chung, chép từ data_raw/ trong repo
├── transnetv2-pytorch-weights.pth   ← dùng chung, tự tải lần đầu
└── output_demo/
    ├── Vanh/                        ← mỗi người một nhánh
    │   ├── v001/                    ← mỗi cấu hình một version
    │   │   ├── progress.json
    │   │   ├── zips/      AIC2026_frames_001.zip …
    │   │   └── indexes/   beit3.index  clip.index  keyframe_metadata.json
    │   │                  keyframes_list.json  image_mapping.json  fps_map.json
    │   └── v002/                    ← lần chạy với cấu hình khác
    ├── Phat/v001/
    ├── An/v001/
    └── Bang/v001/
```

**Khi nào tăng `VERSION`:** đổi cách cắt · đổi tham số lấy mẫu · đổi model encoder ·
đổi độ phân giải lưu ảnh. Chỉ chạy thêm video vào cùng cấu hình thì **giữ nguyên** —
`progress.json` tự bỏ qua video đã xong.

Ô config sẽ cảnh báo nếu version đang chọn đã có dữ liệu, để bạn không lỡ trộn hai
cấu hình vào cùng một index.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02a – AI LÀM PHẦN NÀO + PHẠM VI TEST   ← Ô DUY NHẤT CẦN SỬA
# ═══════════════════════════════════════════════════════════════════════════════
# %%

# ─── 1. Bạn là ai ────────────────────────────────────────────────────────────
# Đổi đúng tên mình. Mỗi người ghi vào thư mục riêng nên không giẫm chân nhau.
PERSON = "Vanh"          # "Vanh" · "Phat" · "An" · "Bang"

# ─── 2. Lần chạy thứ mấy ─────────────────────────────────────────────────────
# Mỗi lần đổi CÁCH LÀM thì tăng version, đừng ghi đè lên lần trước — có giữ lại
# mới so được hai lần chạy với nhau bằng notebook chấm điểm.
#
# Tăng version khi:
#   · đổi cách cắt          (cách 1 ↔ cách 2)
#   · đổi tham số lấy mẫu   (SECONDS_PER_EXTRA, N_MAX, MARGIN…)
#   · đổi model encoder     (BEiT3 ↔ SigLIP…)
#   · đổi độ phân giải lưu ảnh
#
# KHÔNG cần tăng khi chỉ chạy thêm video vào cùng một cấu hình — progress.json
# tự bỏ qua video đã xong, chạy tiếp là được.
VERSION = "v001"        # "v001" · "v002" · "v003" …

# ─── 3. Chạy video nào ───────────────────────────────────────────────────────
# Liệt kê tên video muốn chạy. Để TRỐNG = lấy từ đầu danh sách TEST_VIDEOS.
VIDEO_LIST = [
    # "L21_V015",   # R1-1  KIS · 4 phi hành gia — video đã đo kỹ ở Q1
    # "L30_V092",   # R1-17 KIS · phông đỏ COVID-19 — video để thử OCR
]

# Giới hạn số video. 0 = không giới hạn.
# Lần đầu để 3 cho nhanh (~30 phút); chạy thông hết chuỗi rồi mới nâng lên.
MAX_VIDEOS = 3

assert VERSION.startswith("v") and VERSION[1:].isdigit(), \
    f'VERSION phải dạng "v001", "v002"… — đang là "{VERSION}"'

# ─── 4. Đường dẫn suy ra từ version ──────────────────────────────────────────
# CẢ NHÓM DÙNG CHUNG một thư mục — cùng chỗ mà cut_frames_1…5 ghi keyframe ra.
#
#   aic26/Frame_Transnet_2026_<version>/
#     zips/       AIC2026_frames_p1_001.zip ...
#     indexes/    beit3.index  clip.index  keyframe_metadata.json
#                 keyframes_list.json  image_mapping.json  fps_map.json
#     features/   ocr/  asr/  caption/  tag/  object/
#     progress_p1.json … progress_p5.json     (do bước cắt ghi)
#     progress_<Tên>.json                     (do notebook này ghi)
#
# PERSON chỉ dùng để tách file tiến độ, không tách thư mục nữa — nếu tách thì
# mỗi người có một index riêng, mà index phải là một khối duy nhất cho toàn bộ
# dữ liệu mới tìm được xuyên video.
DRIVE_OUT  = f"/content/drive/MyDrive/aic26/Frame_Transnet_2026_{VERSION}"

# Dùng chung, nằm NGOÀI thư mục riêng để 4 người khỏi tải trọng số 4 lần
SHARED_DIR = "/content/drive/MyDrive/aic26"

print(f"[Person]   {PERSON}")
print(f"[Version]  {VERSION}")
print(f"[Output]   {DRIVE_OUT}")
print(f"[Videos]   " + (f"custom {len(VIDEO_LIST)}" if VIDEO_LIST else "head of list")
      + (f" · limit {MAX_VIDEOS}" if MAX_VIDEOS else " · no limit"))

# Cảnh báo nếu thư mục đã có dữ liệu — tránh trộn hai cấu hình vào cùng một chỗ
import os as _os
if _os.path.exists(f"{DRIVE_OUT}/progress.json"):
    import json as _json
    with open(f"{DRIVE_OUT}/progress.json") as _f:
        _p = _json.load(_f)
    _n = len(_p.get("done_videos", []))
    if _n:
        print()
        print(f"[!] {VERSION} already has {_n} videos cut and {len(_p.get('completed_zips', []))} zips.")
        print("    Continuing will skip those - fine if you are only adding videos.")
        print(f"    If you CHANGED THE METHOD, bump VERSION - do not mix two configs")
        print( "    into one index.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install Dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install] Installing baseline dependencies...")
_pip("--upgrade", "transnetv2-pytorch==1.0.5", "ffmpeg-python")
_pip("open_clip_torch")
_pip("faiss-cpu")   # faiss-gpu trên PyPI dừng ở 2022, không cài được
_pip("tqdm")
_pip("transformers>=4.40.0", "Pillow", "opencv-python-headless")
_pip("gdown")
# BEiT3 (unilm) — dependency thật, verify bằng cách tự cài + test trong sandbox
# trước khi đưa vào đây. deepspeed KHÔNG cần cho inference (verify: chỉ dùng
# trong run_beit3_finetuning.py, không import trong modeling_finetune.py).
_pip("torchscale==0.2.0", "timm==0.4.12", "torchmetrics==0.7.3")
_pip("tensorboardX", "ftfy", "blobfile", "sentencepiece")
# setuptools<81: torchmetrics 0.7.3 dùng pkg_resources (deprecated, bị xóa
# khỏi setuptools>=81) — verify lỗi thật này bằng cách tự chạy import và bắt
# ModuleNotFoundError trong sandbox trước khi thêm dòng pin này.
_pip("setuptools<81")
print("[Install] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install + Config
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys, gc, os, json, glob, zipfile, shutil, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict, Counter
import textwrap

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
def _uninstall(*args):
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-q", "-y", *args])

# Uninstall conflicting paddle versions trước
_uninstall("paddlepaddle-gpu", "paddlepaddle", "paddleocr")

print("[Install] Installing dependencies...")
_pip("open_clip_torch", "transformers>=4.40.0", "opencv-python-headless")
_pip("paddlepaddle")        # CPU – đủ cho OCR
_pip("paddleocr<3.0")       # pin 2.x API, stable — dùng CHỈ để detect box
_pip("vietocr")             # recognition riêng cho tiếng Việt — xem Section 03
_pip("ultralytics")
_pip("einops", "timm")

# ── Fix: broken/mixed Pillow install (ImportError: is_directory) ──────────────
# Root cause THẬT: một trong các package vừa cài (paddlepaddle/paddleocr/
# vietocr) làm Pillow bị cài LẪN giữa 2 version — PIL/ImageFont.py từ 1 bản,
# PIL/_util.py từ bản khác → ImageFont.py import is_directory nhưng _util.py
# hiện tại không có hàm đó.
# Fix trước (Pillow<10, theo running.py) SAI ở đây vì 2 lý do:
#   1. running.py gốc dùng repo fork riêng (bmd1905/vietnamese-ocr, clone qua
#      git), KHÁC với package `vietocr` chuẩn trên PyPI đang dùng ở đây —
#      constraint Pillow<10 của họ không transfer đúng sang pipeline này.
#   2. Colab hiện tại chạy Python 3.12 — Pillow<10 có thể không có wheel sẵn
#      cho 3.12, khiến "pip install" trả về thành công nhưng cài ra bản hỏng.
# Fix đúng: uninstall sạch rồi force-reinstall 1 bản Pillow DUY NHẤT (không ép
# version cũ), làm bước CUỐI trong Section 00 để không package nào sau đó
# ghi đè/làm lẫn lại.
_uninstall("Pillow", "pillow")
_pip("--force-reinstall", "--no-cache-dir", "Pillow")
print("[Install] Done\n")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Eval] Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"[Eval] VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

DRIVE_MOUNT = "/content/drive"
if not os.path.ismount(DRIVE_MOUNT):
    from google.colab import drive; drive.mount(DRIVE_MOUNT)

OUTPUT_ROOT  = DRIVE_OUT
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"   # ← Part 2 Section 13b dùng path này cho ASR
INDEX_DIR    = f"{OUTPUT_ROOT}/indexes"
ZIP_DIR      = f"{OUTPUT_ROOT}/zips"
EVAL_DIR     = "/content/eval_frames"
EVAL_OUT     = "/content/eval_outputs"
shutil.rmtree(EVAL_DIR, ignore_errors=True)   # xóa stale files từ crashed runs
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_OUT).mkdir(exist_ok=True)

N_EVAL_FRAMES = 2000   # ← tăng từ 300 lên 2000

def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()

# ── Vietnamese-compatible font (fix □ tofu boxes in matplotlib text) ───────────
# Root cause: DejaVu Sans Mono (matplotlib's default "monospace" family) lacks
# glyphs for precomposed Vietnamese vowels with combining diacritics. Noto Sans
# has full Latin Extended Additional coverage (which Vietnamese diacritics use).
# Installed via apt (fonts-noto-core) — more reliable on Colab than a raw URL.
try:
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-core"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60)
    import matplotlib.font_manager as _fm
    _noto = glob.glob("/usr/share/fonts/**/NotoSans-Regular.ttf", recursive=True)
    if _noto:
        _fm.fontManager.addfont(_noto[0])
        _VN_FONT = _fm.FontProperties(fname=_noto[0]).get_name()
        plt.rcParams["font.family"] = _VN_FONT
        print(f"[Font] Vietnamese-compatible font registered: {_VN_FONT} ({_noto[0]})")
    else:
        _VN_FONT = "DejaVu Sans"
        print("[Font] WARNING: Noto Sans not found, falling back to DejaVu Sans")
except Exception as e:
    _VN_FONT = "DejaVu Sans"
    print(f"[Font] WARNING: font install failed ({e}), falling back to DejaVu Sans")

# ── JSON results helper (dùng chung cho mọi section) ───────────────────────────
def save_json_results(name: str, data: dict) -> str:
    """Lưu {frame_path_or_name: result} → EVAL_OUT/{name}_results.json"""
    path = os.path.join(EVAL_OUT, f"{name}_results.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    print(f"  [JSON] {name}: {len(data)} entries → {path}")
    return path

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive  ← PHẢI mount TRƯỚC mọi thứ liên quan /content/drive
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os
DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Config & Dirs  ← SAU khi Drive đã mount
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import sys, json, glob, time, math, zipfile, shutil, subprocess as _sp
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─── MODE ────────────────────────────────────────────────────────────────────
# "TEST"  → chỉ xử lý TEST_VIDEOS (~83 videos, đủ để kiểm thử pipeline)
# "FULL"  → toàn bộ ~1478 videos (production run)
MODE = "TEST"   # ← đổi thành "FULL" khi cần chạy thật

# ─── PATHS (path cứng, không đổi) ────────────────────────────────────────────
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
CKPT_TNV2    = f"{SHARED_DIR}/transnetv2-pytorch-weights.pth"
LOCAL_KF_DIR = "/content/kf_buffer"

# ─── TEST VIDEOS ──────────────────────────────────────────────────────────────
# Đối chiếu lại với All_Queries.docx (bản đầy đủ, verify từng câu qua raw XML,
# không dựa vào comment cũ). Round 1 (25 câu) + Round 2 (30 câu) đã full fill,
# Round 3 CHƯA có nội dung trong docx này (chỉ có tiêu đề, không có query nào).
#
# Round 1 GT videos — verify từng câu, KHỚP 100% với docx (24/25 câu có video;
# R1-21 không có video xác nhận — thay vào đó dùng K05_V006 làm benchmark thay
# thế, đúng như docx ghi chú: "Vid gần nhất giống mô tả... K05_V006 (khác mỗi
# ở chỗ không phải "bọ" mà là "chim")... nếu muốn lấy benchmark cho câu này
# thì lấy vid đó làm truth value")
_GT_L = {
    "L21_V015",  # R1-1 KIS: 4 phi hành gia mặc áo đen / tàu vũ trụ tư nhân
    "L21_V027",  # R1-14 KIS: điêu khắc cát, khói hồng
    "L21_V029",  # R1-2 KIS: đàn hổ con quý hiếm miền Nam
    "L22_V022",  # R1-23 KIS: thị trấn ven biển / cá mập (phim Spielberg 1975)
    "L22_V030",  # R1-8 KIS: lễ hội ẩm thực Nhật, cô bé đeo bạch tuộc đỏ
    "L23_V007",  # R1-24 KIS: đua xe đạp, 3 tay đua đồng phục trắng-vàng-xanh
    "L23_V017",  # R1-25 KIS: flycam đua xe đạp, áo xanh dương trắng vượt lên
    "L24_V033",  # R1-16 TRAKE: múa lân vàng đen trắng / lân quay cột
    "L25_V058",  # R2-21 QA: bài tập vận dụng, khoảng cách AB 2 chữ số, đáp C
    "L26_V004",  # R1-20 KIS: panna cotta 3 ly / nho đỏ bạc hà
    "L26_V056",  # R1-6 KIS: gỏi cuốn chay, bánh tráng vàng tím (video đúng)
    "L26_V072",  # R1-18 TRAKE: cắt nấm → củ năng → đậu hũ → bếp
    "L26_V178",  # R1-22 QA: bánh ít trần, 200g thịt nạc xay
    "L26_V194",  # R1-4 TRAKE: bột → măng tây → dầu → chiên
    "L26_V200",  # R1-12 KIS: trang trí bánh rán, chocolate + chuối + dâu
    "L26_V385",  # R1-6 KIS: gỏi cuốn chay (video tìm được, dù L26_V056 đúng hơn)
    "L27_V010",  # R1-19 QA: đình thần Nguyễn Trung Trực Kiên Giang / 2 câu thơ
    "L27_V013",  # R1-9 KIS: thu hoạch dứa miền Tây, bà cụ + cô gái áo hồng + ghe xanh
    "L27_V014",  # R1-5 KIS: cho dê ăn, áo thun trắng + áo kẻ sọc tím
    "L29_V023",  # R1-7 KIS: chim đầu đen ánh xanh mắt đỏ, rừng Nam Bộ
    "L30_V014",  # R2-11 KIS: phụ nữ lớn tuổi đọc sách / bìa ảnh cụ lão
    "L30_V017",  # R1-10 KIS: 3 người chơi nhạc cụ kim loại tròn / kệ sách
    "L30_V040",  # R2-18 KIS: mèo trong phòng, tô kim loại úp, bảng tên "Nà Ní"
    "L30_V057",  # R1-11 KIS: mảnh bìa đổ bóng thành chân dung / mũ lưỡi trai đen
    "L30_V072",  # R1-15 QA: câu lạc bộ FANA trao quà Xã Giang Ly Khánh Hòa
    "L30_V092",  # R1-17 KIS: trao quà từ thiện bệnh viện, phông đỏ COVID-19
    "L30_V095",  # R1-13 KIS: vệ sinh máy ảnh, ống kính + khăn tím hồng + tăm bông
}

# Round 2 GT videos (K-series = dataset batch K). Verify 100% với docx —
# 27/30 câu có video xác nhận (R2-16, R2-19 explicit "chưa tìm được đáp án"
# trong docx → KHÔNG có GT, cố tình bỏ qua. R2-12, R2-23 KHÔNG có video ID
# xác nhận trong docx dù trước đây từng nhớ nhầm là có → xem _LEGACY_UNCONFIRMED)
_GT_K = {
    "K01_V009",  # R2-4 KIS: cứu hộ lũ, cụ bà đội nón lá trên xuồng cam
    "K01_V018",  # R2-15 KIS: cô gái phỏng vấn + Deadpool + múa đao
    "K02_V005",  # R2-5 KIS + R2-22 TRAKE: robot vẽ tranh + sách MILK / khám xét
    "K03_V019",  # R2-1 KIS: nhiều người áo cờ đỏ sao vàng trước biểu tượng con cua
    "K03_V022",  # R2-29 KIS: cảng biển, cầu dài ra biển, người đội nón tai bèo
    "K03_V023",  # R2-8 KIS: cầu vòm thép đỏ → ca nô vệt sóng tròn
    "K05_V013",  # R2-24 KIS: ngư dân Campuchia nâng cá lớn nguy cấp
    "K05_V018",  # R2-25 KIS: robot 4 chân bánh xe trên đồng cỏ + bò
    "K05_V025",  # R2-7 KIS: người tuyết khổng lồ đội mũ + bình gốm băng tuyết
    "K06_V010",  # R2-20 KIS: cào muối đồng, đoàn người, "Gừng cay muối mặn"
    "K07_V025",  # R2-10 TRAKE: xưởng lắp ráp robot + công nhân + cửa ô tô
    "K07_V030",  # R2-17 KIS: phụ nữ đội nón len đeo kính "Happy New Year"
    "K09_V007",  # R2-30 KIS: kị sĩ cưỡi ngựa đồng cỏ xanh / cờ đỏ
    "K09_V012",  # R2-26 KIS: phỏng vấn Trump + đoàn người ven biển vác bao trắng
    "K09_V015",  # R2-13 KIS: bánh giá 9€ và 4.80€ / sản xuất 12 cái/tầng
    "K09_V018",  # R2-6 KIS: đài phun nước trăm ngọn nến / phụ nữ bên ly nến
    "K12_V001",  # R2-9 KIS: cá nhám 211kg treo trên xe cẩu
    "K14_V027",  # R2-2 KIS: lễ hội đèn lồng, hanbok, đèn lồng khổng lồ hình người
    "K15_V010",  # R2-27 KIS: họp giá điện, màn hình LED chữ trắng nền xanh
    "K16_V010",  # R2-14 KIS: icon hình não 5×10 / thống kê tác dụng phụ thuốc
    "K17_V003",  # R2-3 QA: bản đồ Việt Nam trái cây / Khu Du Lịch Quốc Gia Mộc Châu
    "K18_V002",  # R2-28 KIS: nhiều loài cá cảnh, cá đĩa, cá rồng, cá la hán
    "K05_V006",  # R1-21 BENCHMARK THAY THẾ (không phải GT gốc): docx ghi rõ
                 # câu gốc "năm ngoái BTC nói lỗi", K05_V006 là video gần nhất
                 # (khác "bọ" → "chim") được docx đề xuất dùng làm truth value
                 # nếu cần benchmark cho câu này.
}

# ⚠ LEGACY UNCONFIRMED — trước đây từng coi là GT nhưng All_Queries.docx (bản
# đầy đủ, verify qua raw XML) KHÔNG có video ID cho các câu này:
#   K04_V013 (từng gán cho R2-23) — R2-23 trong docx mới KHÔNG có dòng
#     "Kxx_Vxxx,timestamp" nào, chỉ có mô tả + nhận xét, không có đáp án.
#   K07_V019 (từng gán cho R2-12) — tương tự, R2-12 không có video ID.
#   L26_V176, L29_V020 (từng gán cho R3-1, R3-2) — docx này DỪNG NGAY tại
#     tiêu đề "Round 3", không có nội dung Round 3 nào cả, nên không thể
#     verify lại 2 video này từ nguồn hiện tại.
# Quyết định: KHÔNG xóa (đã cắt sẵn trong 95 video hiện có, xóa cũng không
# undo được), nhưng KHÔNG coi là GT đã verify — giữ trong diversity pool.
_LEGACY_UNCONFIRMED = {
    "K04_V013", "K07_V019", "L26_V176", "L29_V020",
}

# Diversity videos (NON-GT) — mở rộng để đạt ~25-30GB frame output (yêu cầu),
# thay vì chỉ 1-3 video/batch như trước (→ 8GB/95 video). Mở rộng lên
# K01-K18 × V001-V008 (144 video) + L21-L30 × V001-V010 (100 video) = 244 mới.
# Ước tính: 8GB/95 video ≈ 86MB frame-output/video trung bình → ~300 video
# tổng (GT+diversity) × 86MB ≈ 25.8GB, khớp mục tiêu 25-30GB. Video nào không
# tồn tại trên Drive sẽ tự động bị skip + WARN (đã có sẵn cơ chế ở Section 03).
_DIVERSITY = {
    "K01_V001", "K01_V002", "K01_V003", "K01_V004", "K01_V005", "K01_V006", "K01_V007", "K01_V008",
    "K02_V001", "K02_V002", "K02_V003", "K02_V004", "K02_V005", "K02_V006", "K02_V007", "K02_V008",
    "K03_V001", "K03_V002", "K03_V003", "K03_V004", "K03_V005", "K03_V006", "K03_V007", "K03_V008",
    "K04_V001", "K04_V002", "K04_V003", "K04_V004", "K04_V005", "K04_V006", "K04_V007", "K04_V008",
    "K05_V001", "K05_V002", "K05_V003", "K05_V004", "K05_V005", "K05_V006", "K05_V007", "K05_V008",
    "K06_V001", "K06_V002", "K06_V003", "K06_V004", "K06_V005", "K06_V006", "K06_V007", "K06_V008",
    "K07_V001", "K07_V002", "K07_V003", "K07_V004", "K07_V005", "K07_V006", "K07_V007", "K07_V008",
    "K08_V001", "K08_V002", "K08_V003", "K08_V004", "K08_V005", "K08_V006", "K08_V007", "K08_V008",
    "K09_V001", "K09_V002", "K09_V003", "K09_V004", "K09_V005", "K09_V006", "K09_V007", "K09_V008",
    "K10_V001", "K10_V002", "K10_V003", "K10_V004", "K10_V005", "K10_V006", "K10_V007", "K10_V008",
    "K11_V001", "K11_V002", "K11_V003", "K11_V004", "K11_V005", "K11_V006", "K11_V007", "K11_V008",
    "K12_V001", "K12_V002", "K12_V003", "K12_V004", "K12_V005", "K12_V006", "K12_V007", "K12_V008",
    "K13_V001", "K13_V002", "K13_V003", "K13_V004", "K13_V005", "K13_V006", "K13_V007", "K13_V008",
    "K14_V001", "K14_V002", "K14_V003", "K14_V004", "K14_V005", "K14_V006", "K14_V007", "K14_V008",
    "K15_V001", "K15_V002", "K15_V003", "K15_V004", "K15_V005", "K15_V006", "K15_V007", "K15_V008",
    "K16_V001", "K16_V002", "K16_V003", "K16_V004", "K16_V005", "K16_V006", "K16_V007", "K16_V008",
    "K17_V001", "K17_V002", "K17_V003", "K17_V004", "K17_V005", "K17_V006", "K17_V007", "K17_V008",
    "K18_V001", "K18_V002", "K18_V003", "K18_V004", "K18_V005", "K18_V006", "K18_V007", "K18_V008",
    "L21_V001", "L21_V002", "L21_V003", "L21_V004", "L21_V005", "L21_V006", "L21_V007", "L21_V008", "L21_V009", "L21_V010",
    "L22_V001", "L22_V002", "L22_V003", "L22_V004", "L22_V005", "L22_V006", "L22_V007", "L22_V008", "L22_V009", "L22_V010",
    "L23_V001", "L23_V002", "L23_V003", "L23_V004", "L23_V005", "L23_V006", "L23_V007", "L23_V008", "L23_V009", "L23_V010",
    "L24_V001", "L24_V002", "L24_V003", "L24_V004", "L24_V005", "L24_V006", "L24_V007", "L24_V008", "L24_V009", "L24_V010",
    "L25_V001", "L25_V002", "L25_V003", "L25_V004", "L25_V005", "L25_V006", "L25_V007", "L25_V008", "L25_V009", "L25_V010",
    "L26_V001", "L26_V002", "L26_V003", "L26_V004", "L26_V005", "L26_V006", "L26_V007", "L26_V008", "L26_V009", "L26_V010",
    "L27_V001", "L27_V002", "L27_V003", "L27_V004", "L27_V005", "L27_V006", "L27_V007", "L27_V008", "L27_V009", "L27_V010",
    "L28_V001", "L28_V002", "L28_V003", "L28_V004", "L28_V005", "L28_V006", "L28_V007", "L28_V008", "L28_V009", "L28_V010",
    "L29_V001", "L29_V002", "L29_V003", "L29_V004", "L29_V005", "L29_V006", "L29_V007", "L29_V008", "L29_V009", "L29_V010",
    "L30_V001", "L30_V002", "L30_V003", "L30_V004", "L30_V005", "L30_V006", "L30_V007", "L30_V008", "L30_V009", "L30_V010",
} | _LEGACY_UNCONFIRMED

# Hợp nhất: GT (~50 videos, verify 100% từ docx) + Diversity (~244+4 videos)
TEST_VIDEOS: set[str] = _GT_L | _GT_K | _DIVERSITY

print(f"[Mode] MODE = {MODE}")
print(f"[Mode] GT videos: {len(_GT_L)+len(_GT_K)} | Diversity: {len(_DIVERSITY)} | Total TEST set: {len(TEST_VIDEOS)}")

# ─── TRANSNETV2 ──────────────────────────────────────────────────────────────
TNV2_W        = 48
TNV2_H        = 27
CUT_THR       = 0.5
MIN_CUT_DIST  = 12
MIN_SCENE_LEN = 8
TNV2_CHUNK    = 200
TNV2_OVERLAP  = 25
TNV2_GDRIVE_ID = "1K7tpO6NMGVqCF30mwuyR-i2HZMA7Fsrg"

# ─── FRAME SELECTION (Paper Alg.1) ───────────────────────────────────────────
N_SAMPLE_PER_SCENE = 4
DEDUP_THR     = 0.90
JPEG_QUALITY  = 90

# ─── ZIP ─────────────────────────────────────────────────────────────────────
ZIP_EVERY_GB  = 2.0
ZIP_DIR       = f"{OUTPUT_ROOT}/zips"
INDEX_DIR     = f"{OUTPUT_ROOT}/indexes"
PROGRESS_FILE = f"{OUTPUT_ROOT}/progress_{PERSON}.json"   # tách theo người

# ─── MODELS ──────────────────────────────────────────────────────────────────
# BEiT3-large (Wang et al., CVPR 2023 — chính xác model [51] paper 2504.08384
# cite cho nhánh "fine-grained"). Checkpoint tốt nhất public cho retrieval:
# beit3_large_patch16_384_coco_retrieval.pth — 675M params, fine-tuned COCO
# retrieval, ITC dual-encoder. Verify TOÀN BỘ bằng cách tự clone source thật
# từ microsoft/unilm, cài dependency, patch lỗi torch._six, tải checkpoint
# thật (1.29GB) và chạy forward pass thành công (output 1024-dim, khớp
# encoder_embed_dim khai báo trong _get_large_config của chính unilm).
BEIT3_CKPT_URL   = "https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth"
BEIT3_SPM_URL    = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"
BEIT3_MODEL_NAME = "beit3_large_patch16_384_retrieval"
BEIT3_INPUT_SIZE = 384
BEIT3_BATCH      = 8   # 675M params fp32 — conservative batch cho T4
BEIT3_CODE_DIR   = "/content/unilm_beit3"   # nơi clone source code thật
# ViT-bigG-14 (LAION, laion2b_s39b_b160k) — giữ nguyên theo quyết định trước:
# nhánh "coarse-grained" dùng CLIP tốt nhất khả thi, KHÔNG cần đúng tên [41]
# (paper cite CLIP gốc OpenAI nhưng không chỉ rõ checkpoint, nên chọn bản
# LAION mạnh nhất fit T4 là hợp lý — đã thảo luận và quyết định trước đó).
CLIP_MODEL       = "ViT-bigG-14"
CLIP_PRETRAIN    = "laion2b_s39b_b160k"
CLIP_BATCH       = 16
FAISS_BEIT3_DIM  = 1024  # verify thật: encoder_embed_dim trong _get_large_config
FAISS_CLIP_DIM   = 1280  # bigG-14 embed_dim, verify từ model config chính thức
NEIGHBOR_WINDOW  = 3

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in [OUTPUT_ROOT, ZIP_DIR, INDEX_DIR, LOCAL_KF_DIR, SHARED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"[Config] VIDEO_ROOT  : {VIDEO_ROOT}")
print(f"[Config] OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"[Config] LOCAL buffer: {LOCAL_KF_DIR}")
print(f"[Config] ZIP every   : {ZIP_EVERY_GB} GB")

# ── Load / Init progress ───────────────────────────────────────────────────────
def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        print(f"[Resume] {len(p.get('done_videos',[]))} videos done, "
              f"{p.get('zip_count',0)} zips created.")
        return p
    print("[Resume] No progress file found – starting fresh.")
    return {
        "done_videos": [],
        "zip_count": 0,
        "completed_zips": [],
        "buffer_videos": [],
        "buffer_size_bytes": 0,
        "total_frames_cut": 0,
        "index_done_beit3": [],
        "index_done_clip":   [],
        "index_phase_done":  False,
        # Feature extraction phase (Part 2)
        "done_ocr":     [],
        "done_asr":     [],
        "done_caption": [],
        "done_tags":    [],
        "done_objects": [],
        "features_merged": False,
        "last_updated": None,
    }

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02b – Buffer Recovery (quan trọng: phát hiện videos bị mất khi disconnect)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Khi Colab disconnect, /content/kf_buffer bị xóa.
# Videos đã mark done nhưng chưa kịp zip → keyframes mất.
# Section này phát hiện và đưa chúng về hàng đợi lại.
# %%

def _find_unzipped_videos() -> set[str]:
    """Tìm video names đã có trong các completed_zips."""
    found = set()
    for zname in PROGRESS.get("completed_zips", []):
        zpath = os.path.join(ZIP_DIR, zname)
        if not os.path.exists(zpath): continue
        with zipfile.ZipFile(zpath) as zf:
            for name in zf.namelist():
                parts = name.split("/")
                if parts: found.add(parts[0])
    return found

_zipped_videos = _find_unzipped_videos()
_buffer_lost   = [v for v in PROGRESS.get("buffer_videos", [])
                  if v not in _zipped_videos]

if _buffer_lost:
    print(f"[Recovery] Found {len(_buffer_lost)} buffered videos lost on disconnect:")
    for v in _buffer_lost:
        print(f"  ↩ {v}")
    # Đưa về done_videos để cut lại
    for v in _buffer_lost:
        if v in PROGRESS["done_videos"]:
            PROGRESS["done_videos"].remove(v)
    PROGRESS["buffer_videos"]    = [v for v in PROGRESS["buffer_videos"]
                                    if v not in _buffer_lost]
    PROGRESS["buffer_size_bytes"] = 0
    save_progress(PROGRESS)
    print("[Recovery] Re-queued. They will be re-cut in Section 07.")
else:
    print("[Recovery] Buffer OK - no videos lost.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – List Videos (filter theo MODE)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

def list_all_videos(root: str) -> list[str]:
    vids = sorted(set(
        glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True) +
        glob.glob(os.path.join(root, "*.mp4"))
    ))
    if not vids:
        raise FileNotFoundError(f"No videos found in {root}")
    return vids

_all_raw      = list_all_videos(VIDEO_ROOT)
# Filter theo MODE
if MODE == "TEST":
    ALL_VIDEOS = [v for v in _all_raw if Path(v).stem in TEST_VIDEOS]
    _missing   = TEST_VIDEOS - {Path(v).stem for v in _all_raw}
    if _missing:
        print(f"[TEST] WARN: {len(_missing)} TEST_VIDEOS not found on Drive:")
        for m in sorted(_missing): print(f"  ✗ {m}")
else:  # FULL
    ALL_VIDEOS = _all_raw

# ─── Lọc theo VIDEO_LIST ở SECTION 02a ───────────────────────────────────────
if VIDEO_LIST:
    _wanted = set(VIDEO_LIST)
    ALL_VIDEOS = [v for v in ALL_VIDEOS if Path(v).stem in _wanted]
    _missing = _wanted - {Path(v).stem for v in ALL_VIDEOS}
    if _missing:
        print(f"[Select]   WARN not found on Drive: {sorted(_missing)}")
    print(f"[Select]   {len(ALL_VIDEOS)} videos from VIDEO_LIST")
else:
    ALL_VIDEOS = sorted(ALL_VIDEOS)
    print(f"[Select]   VIDEO_LIST empty -> taking from head of {len(ALL_VIDEOS)} videos")

done_set  = set(PROGRESS["done_videos"])
todo_list = [v for v in ALL_VIDEOS if Path(v).stem not in done_set]

if MAX_VIDEOS and len(todo_list) > MAX_VIDEOS:
    todo_list = todo_list[:MAX_VIDEOS]
    print(f"[Limit] Running only {MAX_VIDEOS} videos: {[Path(v).stem for v in todo_list]}")
    print( "           Set MAX_VIDEOS = 0 in SECTION 02a to run everything.")

total_gb = sum(os.path.getsize(v) for v in ALL_VIDEOS) / 1024**3
done_gb  = sum(os.path.getsize(v) for v in ALL_VIDEOS
               if Path(v).stem in done_set) / 1024**3

print(f"\n[Videos] MODE={MODE} | Total: {len(ALL_VIDEOS)} | Done: {len(done_set)} | Remaining: {len(todo_list)}")
print(f"[Size]   Total: {total_gb:.1f} GB | Done: {done_gb:.1f} GB")
print(f"[Sample] First 5 remaining:")
for v in todo_list[:5]:
    print(f"  {Path(v).stem}  ({os.path.getsize(v)/1024**2:.0f} MB)")

if not todo_list:
    print("\n[OK] All videos cut. Go to Section 09 to build the index.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03a – Chuẩn bị cho ASR (thay cho bước unzip keyframe)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# ASR chạy ở mức VIDEO, không cần keyframe — đây là nhánh duy nhất trên sơ đồ
# offline không đi qua hộp `Keyframes`. Phòng lab gốc lấy `vid_set` từ các frame
# đã giải nén; ở đây lấy thẳng từ danh sách video của mình.

vid_set = {Path(v).stem for v in todo_list}
print(f"[ASR] {len(vid_set)} videos in {PERSON}'s part: {sorted(vid_set)[:8]}"
      + (" …" if len(vid_set) > 8 else ""))

# Dùng để gán mỗi đoạn lời thoại về các keyframe phủ khoảng thời gian đó.
# Nếu đã chạy bước cắt + index thì có; chưa có thì ASR vẫn chạy, chỉ là không
# ánh xạ được sang keyframe.
META_BY_NAME: dict[str, dict] = {}
_meta_p = os.path.join(INDEX_DIR, "keyframe_metadata.json")
if os.path.exists(_meta_p):
    with open(_meta_p) as f:
        META_BY_NAME = {m["name"]: m for m in json.load(f)}
    print(f"[ASR] Loaded {len(META_BY_NAME)} keyframes for second -> frame mapping")
else:
    print("[ASR] No keyframe_metadata.json - skipping the keyframe mapping step.")
    print("      Transcripts are still extracted in full, only the frame column is missing.")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

---

## Phòng lab — đánh giá Whisper

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03b – Whisper ASR Evaluation
# ─────────────────────────────────────────────────────────────────────────────
# Architecture: Whisper large-v3 (Radford et al., OpenAI, 2022) — encoder-decoder
#   Transformer, ~1.55B params, 680k giờ multilingual + multitask supervised data.
# Config: bám sát ĐÚNG Part 2 Section 15 (production) — cùng model size, cùng
#   audio extraction command (ffmpeg → 16kHz mono pcm_s16le), cùng transcribe
#   params (language="vi", task="transcribe", word_timestamps=False).
# Scope: ASR chạy ở mức VIDEO (không phải per-frame) nên không cần sample 2000
#   frames — lấy 8 video (trong 5-10 theo yêu cầu) từ chính vid_set đã dùng cho
#   các section khác (OCR/Vintern/Color) để evaluation nhất quán trên cùng data.
# Known risk (grounded — SOICT 2024 survey docx, KHÔNG tự suy diễn thêm):
#   một team báo cáo nguyên văn "we have developed a method to overcome the
#   hallucinations often encountered in Whisper large models with Vietnamese
#   pretraining" — nghĩa là Whisper large-v3 + tiếng Việt là rủi ro hallucination
#   ĐÃ ĐƯỢC document bởi ít nhất 1 team SOICT, nhưng docx không cho biết chi tiết
#   fix của họ. Section này KHÔNG tự chế fix — chỉ show raw output để bạn tự
#   review bằng mắt (giống cách bạn review được lỗi lặp từ ở Vintern).
# ═══════════════════════════════════════════════════════════════════════════════
# %%

print("[ASR] Installing Whisper (mirrors Part 2 Section 13a install list)...")
_pip("openai-whisper", "ffmpeg-python")

import whisper, subprocess as _sp

ASR_MODEL_SIZE  = "large-v3"                    # giống hệt Part 2
N_ASR_VIDEOS    = min(8, len(vid_set))          # trong khoảng 5-10 theo yêu cầu
ASR_TEST_VIDEOS = sorted(vid_set)[:N_ASR_VIDEOS]
print(f"[ASR] Loading Whisper {ASR_MODEL_SIZE}... test videos: {ASR_TEST_VIDEOS}")
whisper_model = whisper.load_model(ASR_MODEL_SIZE, device=DEVICE)
if DEVICE == "cuda":
    print(f"[ASR] VRAM: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

def _frames_for_segment(video_frames: list[dict], start_s: float, end_s: float) -> list[str]:
    """Map ASR segment time range → overlapping keyframe names.
    ±0.5s tolerance — giống hệt Part 2 Section 15 _frames_for_segment()."""
    return [m["name"] for m in video_frames
            if start_s - 0.5 <= m.get("timestamp_ms", 0)/1000 <= end_s + 0.5]

asr_json_results = {}
asr_viz_rows = []   # (video_name, full_text, first_segments)

for video_name in ASR_TEST_VIDEOS:
    matches = (glob.glob(f"{VIDEO_ROOT}/**/{video_name}.mp4", recursive=True) +
               glob.glob(f"{VIDEO_ROOT}/{video_name}.mp4"))
    if not matches:
        print(f"  [ASR] WARN: {video_name}.mp4 not found on Drive, skip")
        continue

    audio_tmp = f"/tmp/{video_name}_audio.wav"
    print(f"  [ASR] Extracting audio: {video_name}...")
    # Cùng ffmpeg command với Part 2 Section 15 — 16kHz mono, pcm_s16le
    r = _sp.run(["ffmpeg", "-y", "-i", matches[0], "-vn",
                 "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_tmp],
               capture_output=True, timeout=300)
    if r.returncode != 0 or not os.path.exists(audio_tmp):
        print(f"  [ASR] WARN: audio extract failed for {video_name}")
        continue

    print(f"  [ASR] Transcribing {video_name}...")
    t0 = time.time()
    result = whisper_model.transcribe(audio_tmp, language="vi",
                                      task="transcribe", word_timestamps=False)
    if os.path.exists(audio_tmp): os.remove(audio_tmp)

    # META_BY_NAME (từ Section 01) chứa frame metadata với timestamp_ms + video
    video_frames = sorted(
        (m for m in META_BY_NAME.values() if m.get("video") == video_name),
        key=lambda m: m.get("frame_idx", 0)
    )
    segments = []
    for seg in result.get("segments", []):
        text = seg["text"].strip()
        if not text: continue
        segments.append({
            "start_s": round(seg["start"], 2), "end_s": round(seg["end"], 2),
            "text": text,
            "frames": _frames_for_segment(video_frames, seg["start"], seg["end"]),
        })

    asr_json_results[video_name] = {
        "full_text": result.get("text", ""), "segments": segments
    }
    asr_viz_rows.append((video_name, result.get("text", ""), segments[:6]))
    print(f"  [ASR] ✓ {video_name}: {len(segments)} segments in {time.time()-t0:.0f}s")

# Viz: mỗi video 1 panel — full transcript (300 ký tự đầu) + 6 segment đầu kèm timestamp
if asr_viz_rows:
    fig, axes = plt.subplots(len(asr_viz_rows), 1, figsize=(16, 3.4 * len(asr_viz_rows)))
    _axes_asr = np.atleast_1d(axes)
    for i, (vname, full_text, segs) in enumerate(asr_viz_rows):
        seg_lines = "\n".join(f"  [{s['start_s']:.1f}-{s['end_s']:.1f}s] {s['text']}"
                              for s in segs)
        panel = (f"[{vname}]  full_text ({len(full_text)} chars):\n{full_text[:300]}\n\n"
                f"First {len(segs)} segments:\n{seg_lines}")
        wrapped = "\n".join(textwrap.wrap(panel, 110,
                                          replace_whitespace=False, drop_whitespace=False))
        _axes_asr[i].text(0.01, 0.95, wrapped, transform=_axes_asr[i].transAxes,
                          va="top", ha="left", fontsize=7.5,
                          bbox=dict(boxstyle="round,pad=0.4",
                                    facecolor="#f0f4ff", edgecolor="#bbb"))
        _axes_asr[i].axis("off")

    plt.suptitle(f"Whisper {ASR_MODEL_SIZE} ASR — {len(asr_viz_rows)} videos "
                "(language=vi, task=transcribe — cùng config Part 2 production)",
                fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{EVAL_OUT}/asr_eval.png", dpi=90, bbox_inches="tight")
    plt.show()
    plt.close(fig)   # release figure memory — prevents OOM crash across sections

save_json_results("asr", asr_json_results)
print(f"[ASR] Done. {len(asr_json_results)}/{len(ASR_TEST_VIDEOS)} videos transcribed")

del whisper_model; _free()

---

## Chạy thật — trích lời thoại cho phần video của mình

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 13a – Install Feature Extraction Dependencies (chạy 1 lần)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
import subprocess, sys, gc
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install-feat] Installing feature extraction dependencies...")
_pip("paddlepaddle-gpu", "paddleocr")
_pip("openai-whisper", "ffmpeg-python")
_pip("einops", "timm", "accelerate")    # Vintern dependencies
_pip("ultralytics")                      # YOLOv8
try:
    _pip("git+https://github.com/xinyu1205/recognize-anything.git", "--quiet")
    _RAM_INSTALL_OK = True
    print("[Install-feat] RAM++ installed OK")
except Exception:
    _RAM_INSTALL_OK = False
    print("[Install-feat] RAM++ not available – will use Vintern tag-prompt fallback")
_pip("transformers>=4.40.0", "faiss-gpu", "Pillow", "opencv-python-headless")
print("[Install-feat] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 13b – Config + Mount + Load Progress
# ═══════════════════════════════════════════════════════════════════════════════
# %%
import os, json, glob, time, zipfile, shutil
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from PIL import Image

DRIVE_MOUNT = "/content/drive"
if not os.path.ismount(DRIVE_MOUNT):
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted.")
else:
    print("[Drive] Already mounted.")

VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
INDEX_DIR    = f"{OUTPUT_ROOT}/indexes"
ZIP_DIR      = f"{OUTPUT_ROOT}/zips"
FEATURES_DIR = f"{OUTPUT_ROOT}/features"
PROGRESS_FILE= f"{OUTPUT_ROOT}/progress.json"
UNZIP_DIR    = "/content/unzip_feat"

for d in [FEATURES_DIR, UNZIP_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Hyperparams ───────────────────────────────────────────────────────────────
ASR_MODEL_SIZE  = "large-v3"   # best quality; dùng "medium" nếu cần nhanh hơn
VINTERN_MODEL   = "5CD-AI/Vintern-1B-v2"
VINTERN_BATCH   = 1            # T4: 1 an toàn nhất (2GB/frame bfloat16)
YOLO_MODEL      = "yolov8l.pt"
YOLO_CONF       = 0.30
RAM_IMG_SIZE    = 384
OCR_LANG        = "vi"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"[System] VRAM free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB total")

def _free_gpu():
    """Release GPU memory và run garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: p = json.load(f)
    else:
        p = {}
    for key in ["done_ocr","done_asr","done_caption","done_tags","done_objects","completed_zips"]:
        p.setdefault(key, [])
    p.setdefault("features_merged", False)
    return p

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

FULL_META_PATH = os.path.join(INDEX_DIR, "keyframe_metadata.json")
if not os.path.exists(FULL_META_PATH):
    raise FileNotFoundError("keyframe_metadata.json not found – run Part 1 first")
with open(FULL_META_PATH) as f: FULL_META = json.load(f)
VIDEO_FRAMES: dict[str, list[dict]] = defaultdict(list)
for m in FULL_META:
    VIDEO_FRAMES[m["video"]].append(m)
for v in VIDEO_FRAMES:
    VIDEO_FRAMES[v].sort(key=lambda x: x["frame_idx"])
print(f"[Meta] {len(FULL_META)} frames across {len(VIDEO_FRAMES)} videos")

# ── Helper: unzip one video from completed zips ────────────────────────────────
def unzip_video(video_name: str) -> bool:
    """Unzip frames of video_name từ completed_zips vào UNZIP_DIR. Returns True nếu found."""
    p = load_progress()
    for zip_name in p.get("completed_zips", []):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): continue
        with zipfile.ZipFile(zip_path) as zf:
            members = [n for n in zf.namelist() if n.startswith(f"{video_name}/")]
            if not members: continue
            shutil.rmtree(os.path.join(UNZIP_DIR, video_name), ignore_errors=True)
            for m in members: zf.extract(m, UNZIP_DIR)
        return True
    return False

def rm_video_buf(video_name: str):
    shutil.rmtree(os.path.join(UNZIP_DIR, video_name), ignore_errors=True)

print(f"[Config] FEATURES_DIR: {FEATURES_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 15 – ASR  (Whisper large-v3, ~3GB VRAM khi active)
# ─────────────────────────────────────────────────────────────────────────────
# GPU POLICY:
#   Load Whisper → process tất cả videos → del whisper → VRAM về 0 → next section.
#   Whisper large-v3 cần ~3GB VRAM. Không load nếu section đã done.
# ═══════════════════════════════════════════════════════════════════════════════
# %%

PROGRESS = load_progress()
todo_asr  = [v for v in VIDEO_FRAMES if v not in PROGRESS["done_asr"]]

if not todo_asr:
    print(f"[SKIP] ASR already complete ({len(PROGRESS['done_asr'])} videos).")
else:
    import whisper, subprocess as _sp
    print(f"[ASR] Loading Whisper {ASR_MODEL_SIZE} (~3GB VRAM) – {len(todo_asr)} videos")
    whisper_model = whisper.load_model(ASR_MODEL_SIZE, device=DEVICE)
    if DEVICE == "cuda":
        used = torch.cuda.memory_allocated() / 1024**3
        print(f"[ASR] Whisper loaded. VRAM used: {used:.1f}GB")

    def _frames_for_segment(frames, start_s, end_s):
        return [m["name"] for m in frames
                if start_s - 0.5 <= m["timestamp_ms"]/1000 <= end_s + 0.5]

    for video_name in todo_asr:
        out_path = os.path.join(FEATURES_DIR, f"{video_name}_asr.json")
        if os.path.exists(out_path):
            PROGRESS["done_asr"].append(video_name); save_progress(PROGRESS); continue
        matches = (glob.glob(f"{VIDEO_ROOT}/**/{video_name}.mp4", recursive=True) +
                   glob.glob(f"{VIDEO_ROOT}/{video_name}.mp4"))
        if not matches:
            print(f"  [ASR] WARN: {video_name}.mp4 not found on Drive, skip"); continue
        audio_tmp = f"/tmp/{video_name}_audio.wav"
        r = _sp.run(["ffmpeg","-y","-i",matches[0],"-vn",
                     "-acodec","pcm_s16le","-ar","16000","-ac","1",audio_tmp],
                    capture_output=True, timeout=300)
        if r.returncode != 0 or not os.path.exists(audio_tmp):
            print(f"  [ASR] WARN: audio extract failed for {video_name}"); continue
        print(f"  [ASR] Transcribing {video_name}..."); t0 = time.time()
        result = whisper_model.transcribe(audio_tmp, language="vi",
                                           task="transcribe", word_timestamps=False)
        if os.path.exists(audio_tmp): os.remove(audio_tmp)
        frames   = VIDEO_FRAMES[video_name]
        segments = []
        for seg in result.get("segments", []):
            text = seg["text"].strip()
            if not text: continue
            segments.append({"start_s": round(seg["start"],2), "end_s": round(seg["end"],2),
                              "text": text,
                              "frames": _frames_for_segment(frames, seg["start"], seg["end"])})
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump({"video": video_name, "language": "vi",
                       "full_text": result.get("text",""), "segments": segments},
                      f, ensure_ascii=False, indent=1)
        PROGRESS["done_asr"].append(video_name); save_progress(PROGRESS)
        print(f"  [ASR] ✓ {video_name}: {len(segments)} segments in {time.time()-t0:.0f}s")

    # ← Unload Whisper: giải phóng ~3GB VRAM trước khi load Vintern
    del whisper_model; _free_gpu()
    print(f"[ASR] Done + Whisper unloaded. VRAM free: "
          f"{torch.cuda.mem_get_info()[0]/1024**3:.1f}GB" if DEVICE=='cuda' else "[ASR] Done.")